Hệ thống gợi ý là gì & Tại sao cần nó?

    Hệ thống gợi ý là hệ thống học máy dự đoán sở thích người dùng và đề xuất các mục (sản phẩm, nội dung) phù hợp.
        10 triệu cuốn sách (Quá tải thông tin / Information Overload)
        -> Bộ lọc thông minh (AI)
        -> Vài cuốn sách bạn thực sự thích

        => Lọc bỏ nhiễu
        => Cá nhân hóa trải nghiệm
        => Tiết kiệm thời gian tìm kiếm

    Ứng dụng thực tế hàng ngày
        Netflix: Gợi ý phim/series dựa trên lịch sử xem.
        Spotify: Playlist "Discover Weekly" cá nhân hóa.
        Amazon: Mục "Khách hàng mua sản phẩm này cũng mua ... "
        YouTube: Video tiếp theo bạn nên xem (Next Up).
        TikTok: Nội dung trên trang "For You".
        Kết luận: Hệ thống gợi ý hiện diện ở mọi nền tảng số lớn.

Quy trình & Thu thập dữ liệu
    Thu thập dữ liệu -> Xử lý dữ liệu -> Xây dựng mô hình -> Đánh giá/Triển khai

    Phân loại dữ liệu đầu vào
        Dữ liệu Tường minh (Explicit)
            · Mô tả: Người dùng chủ động đánh giá.
            · Ví dụ: Rate 5 sao, like/dislike.
            · Đặc điểm: Chính xác nhưng Ít (User lười đánh giá).

        Dữ liệu Ngầm định (Implicit)
            · Mô tả: Suy luận từ hành vi.
            · Ví dụ: Lịch sử xem, click, thời gian đọc.
            · Đặc điểm: Nhiều nhưng Nhiễu (Xem # Thích).

    Ma trận Người dùng - Mục (User-Item Matrix)
                Movie A     Movie B     Movie C     Movie D

        User 1      5

        User 2                  3

        User 3                              4

        User 4

        User 5

        => Tính thưa (Sparsity): vấn đề thực tế hơn 99% ô là trống vì người dùng chỉ tương tác với một phần nhỏ tổng số mục.
        => Nhiệm vụ của hệ thống: Điền vào các ô '?' (dự đoán điểm số) dựa trên dữ liệu hiện có.

    Xử lý dữ liệu: Chuẩn hóa (Normalization)
        Vấn đề: Bias Thang đo 
            · User A (Dễ tính): Thích = 5, Ghet = 4
            · User B (Khó tính): Thích = 3, Ghet = 1
        => Hậu quả: Điểm số thô không phản ánh đúng mức độ yêu thích nếu so sánh ngang hàng.

    Giải pháp: Mean Centering
        r'ui = rui - ru
        rui: Điểm gốc
        ru: Điểm trung bình của user

        Đưa về cùng một mặt bằng so sánh (dương là thích, âm là không thích so với trung bình của chính họ).



Đánh giá sai số dự đoán

    MAE (Mean Absolute Error): Trung bình khoảng cách tuyệt đối giữa giá trị thực và dự đoán.
        MAE = 1/n x |∑(yi - ŷi)|

    RMSE (Root Mean Square Error): Phạt nặng các sai số lớn do phép bình phương.
        RMSE = √1/n x ∑(n; i=1)(yi​−ŷi)^2

Đánh giá danh sách Top-K: Precision & Recall
    Dùng khi hệ thống đề xuất một danh sách (ví dụ: 5 phim) thay vì dự đoán điểm số cụ thể.

    Công thức:
        Precision@K = Số mục liên quan trong top K / K
            Ví dụ:
                Gợi ý: [A, B, C, D, E]
                -> User thực sự thích: [A, C, F, G]
                -> Kết quả trùng (Relevant): 2 phim (A, C)
                -> Precision@5 = 2 / 5=40%        

        Recall@K = Số mục liên quan trong top K /  (Tổng số mục liên quan)        


Ba phương pháp tiếp cận chính
    Lọc cộng tác (Collaborative Filtering - CF)
        Ý tưởng: Người có sở thích giống nhau trong quá khứ sẽ giống nhau trong tương lai -> Hành vi đám đông.
        Triết lý: Không quan tâm nội dung mục, chỉ quan tâm tương tác cộng đồng.

        User-based CF: Tìm người giống bạn. (Users similar to User U)

        Item-based CF: Tìm mục giống mục bạn thích. (Items similar to Item l)




    
    Lọc dựa trên nội dung (Content-based Filtering)
        Ý tưởng: Gợi ý các mục có đặc điểm tương tự mục người dùng đã thích -> Thuộc tính sản phẩm

    Phương pháp lai (Hybrid)
        Ý tưởng: Kết hợp cả hai để tận dụng ưu điểm và giảm thiểu nhược điểm -> Example: Netflix.




In [79]:
import pandas as pd
import numpy as np

bookDf = pd.read_csv('Books.csv')
ratingDf = pd.read_csv('Ratings.csv')
userDf = pd.read_csv('Users.csv')
ratingDf

C:\Users\Ryan\AppData\Local\Temp\ipykernel_6904\188806887.py:4: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  bookDf = pd.read_csv('Books.csv')


,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6
...,...,...,...
1149775,276704,1563526298,9
1149776,276706,0679447156,0
1149777,276709,0515107662,10
1149778,276721,0590442449,10


In [63]:
# Count how many times a book is rated
ratingCountDf = ratingDf.groupby('ISBN').count()
print(type(ratingCountDf)) # Data frame
ratingCountDf

ratingCountDf = ratingCountDf['Book-Rating']
print(type(ratingCountDf)) # Series
ratingCountDf

ratingCountDf = ratingCountDf.reset_index(name='User-Rating-Count')
print(type(ratingCountDf)) # DataFrame
print(len(ratingCountDf))
ratingCountDf

<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.frame.DataFrame'>
340556


,ISBN,User-Rating-Count
0,0330299891,2
1,0375404120,2
2,0586045007,1
3,9022906116,2
4,9032803328,1
...,...,...
340551,cn113107,1
340552,ooo7156103,1
340553,§423350229,1
340554,´3499128624,1


In [ ]:
# Calculate the mean rating score for each book
ratingMeanDf = ratingDf.groupby('ISBN')['Book-Rating'].mean()
print(type(ratingMeanDf)) # Series
ratingMeanDf = ratingMeanDf.reset_index()
print(type(ratingMeanDf)) # DataFrame
print(len(ratingMeanDf))
ratingMeanDf

<class 'pandas.core.series.Series'>
<class 'pandas.core.frame.DataFrame'>
340556


,ISBN,Book-Rating
0,0330299891,3.0
1,0375404120,1.5
2,0586045007,0.0
3,9022906116,3.5
4,9032803328,0.0
...,...,...
340551,cn113107,0.0
340552,ooo7156103,7.0
340553,§423350229,0.0
340554,´3499128624,8.0


In [46]:
# Merge ratingCountDf and ratingMeanDf
ratingAggregateDf = ratingCountDf.merge(ratingMeanDf, on='ISBN')
ratingAggregateDf
ratingCountDf

,ISBN,Book-Rating
0,0330299891,2
1,0375404120,2
2,0586045007,1
3,9022906116,2
4,9032803328,1
...,...,...
340551,cn113107,1
340552,ooo7156103,1
340553,§423350229,1
340554,´3499128624,1


In [82]:
# Merge ratingBookAggregate with bookDf to get more information for a book.
ratingBookAggregateDf = ratingAggregateDf.merge(bookDf, on='ISBN')
print(type(ratingBookAggregateDf)) # dataframe
print(len(ratingBookAggregateDf))
ratingBookAggregateDf.rename(columns={'Book-Rating_x': 'Rating-Count', 'Book-Rating_y': 'Rating-Avg'}, inplace=True)
ratingBookAggregateDf = ratingBookAggregateDf.drop(columns=['Image-URL-M', 'Image-URL-L'])
ratingBookAggregateDf.head()

# Filter to get all the books which has Rating-Count >= 100
popularRatingBookAggregateDf = ratingBookAggregateDf[ratingBookAggregateDf['Rating-Count'] >= 100].reset_index()
print(type(popularRatingBookAggregateDf))
print(len(popularRatingBookAggregateDf))
popularRatingBookAggregateDf.head()

<class 'pandas.core.frame.DataFrame'>
270151
<class 'pandas.core.frame.DataFrame'>
727


,index,ISBN,Rating-Count,Rating-Avg,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S
0,1759,002542730X,171,3.514620,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc,http://images.amazon.com/images/P/002542730X.0...
1,3219,0060008032,104,2.442308,Angels,Marian Keyes,2003,HarperTorch,http://images.amazon.com/images/P/0060008032.0...
2,3484,0060096195,107,4.028037,The Boy Next Door,Meggin Cabot,2002,Avon Trade,http://images.amazon.com/images/P/0060096195.0...
3,3989,006016848X,147,2.693878,"Men Are from Mars, Women Are from Venus: A Pra...",John Gray,1992,HarperCollins Publishers,http://images.amazon.com/images/P/006016848X.0...
4,4122,0060173289,130,3.453846,Divine Secrets of the Ya-Ya Sisterhood : A Novel,Rebecca Wells,1996,HarperCollins,http://images.amazon.com/images/P/0060173289.0...


In [86]:
# Count the rating times of each user
activeUser = ratingDf.groupby('User-ID').size().reset_index(name='User-Rating-Count')
print(type(activeUser)) # DataFrame
activeUser

# Filter to get the users who have rating times >= 100
activeUser = activeUser[activeUser['User-Rating-Count'] >= 100].sort_values(by='User-Rating-Count', ascending=False)
print(type(activeUser))
print(len(activeUser))
activeUser


# Active rating by user
activeUserRatingDf = ratingDf[ratingDf['User-ID'].isin(activeUser['User-ID'])]
print(len(activeUserRatingDf))
print(type(activeUserRatingDf))
activeUserRatingDf


<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
1847
658805
<class 'pandas.core.frame.DataFrame'>


,User-ID,ISBN,Book-Rating
412,276925,0006511929,0
413,276925,002542730X,10
414,276925,0060520507,0
415,276925,0060930934,0
416,276925,0060951303,0
...,...,...,...
1149633,276680,1884910033,0
1149634,276680,1888173408,7
1149635,276680,1888173564,8
1149636,276680,1888173572,0


In [92]:
# the books which has popular rating by active users
polularUserBookRatingDf = activeUserRatingDf[activeUserRatingDf['ISBN'].isin(popularRatingBookAggregateDf['ISBN'])]
print(len(polularUserBookRatingDf)) # 65523
polularUserBookRatingDf

# Merge with bookDf to get more book information
polularUserBookRatingDf = polularUserBookRatingDf.merge(bookDf, on='ISBN')
print(len(polularUserBookRatingDf)) # 65523
polularUserBookRatingDf = polularUserBookRatingDf.drop(columns=['Image-URL-M', 'Image-URL-L'])
polularUserBookRatingDf

65523
65523


,User-ID,ISBN,Book-Rating,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S
0,276925,002542730X,10,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc,http://images.amazon.com/images/P/002542730X.0...
1,276925,0316666343,0,The Lovely Bones: A Novel,Alice Sebold,2002,"Little, Brown",http://images.amazon.com/images/P/0316666343.0...
2,276925,0385504209,8,The Da Vinci Code,Dan Brown,2003,Doubleday,http://images.amazon.com/images/P/0385504209.0...
3,276925,0804106304,0,The Joy Luck Club,Amy Tan,1994,Prentice Hall (K-12),http://images.amazon.com/images/P/0804106304.0...
4,276925,0971880107,0,Wild Animus,Rich Shapero,2004,Too Far,http://images.amazon.com/images/P/0971880107.0...
...,...,...,...,...,...,...,...,...
65518,276680,0440221595,8,The Glass Lake,Maeve Binchy,1996,Dell,http://images.amazon.com/images/P/0440221595.0...
65519,276680,0446670251,0,The Virgin Suicides,Jeffrey Eugenides,1994,Warner Books,http://images.amazon.com/images/P/0446670251.0...
65520,276680,0452283205,7,Falling Angels,Tracy Chevalier,2002,Plume Books,http://images.amazon.com/images/P/0452283205.0...
65521,276680,0679731725,0,The Remains of the Day (Vintage International),Kazuo Ishiguro,1993,Vintage Books USA,http://images.amazon.com/images/P/0679731725.0...


In [93]:
pivot_table = polularUserBookRatingDf.pivot_table(index='Book-Title', columns='User-ID', values='Book-Rating')
pivot_table.head()

pivot_table.fillna(0, inplace=True)
pivot_table.head()

User-ID,254,507,882,1424,1435,1733,1903,2033,2110,2276,...,276018,276463,276680,276925,277427,277478,277639,278137,278188,278418
Book-Title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2nd Chance,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4 Blondes,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A Beautiful Mind: The Life of Mathematical Genius and Nobel Laureate John Nash,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
item_similarity = cosine_similarity(pivot_table)
item_similarity.shape

print(item_similarity[:4])

book = 'Harry Potter and the Chamber of Secrets (Book 2)'
book_index = pivot_table.index.tolist().index(book)
print(f'Index of the book "{book}":', book_index)

book_similarities = item_similarity[book_index]
print(book_similarities)

# Top 5 similar books
sorted_indexes = sorted(range(len(book_similarities)), key=lambda k: book_similarities[k], reverse=True)
for i in sorted_indexes[1:6]:
    print(f'Index: {i}, Similarity Score: {book_similarities[i]:.4f}, Book Title: {pivot_table.index[i]}')

[[1.         0.05891323 0.         ... 0.         0.07039485 0.        ]
 [0.05891323 1.         0.24207864 ... 0.12679146 0.04413729 0.16688272]
 [0.         0.24207864 1.         ... 0.11199938 0.02440714 0.05509168]
 [0.         0.         0.         ... 0.05143611 0.         0.        ]]
Index of the book "Harry Potter and the Chamber of Secrets (Book 2)": 216
[0.04229094 0.05161445 0.06726978 0.         0.         0.01992399
 0.01646777 0.09843386 0.         0.06665909 0.09574041 0.
 0.02685106 0.0543404  0.0514492  0.02203904 0.02040047 0.04444827
 0.09702149 0.06093805 0.09132612 0.08353835 0.05096309 0.13789726
 0.         0.10364696 0.08716438 0.01528515 0.10505786 0.05400098
 0.07396335 0.10614278 0.02798494 0.0914833  0.02203452 0.05350447
 0.08150462 0.03472364 0.0932976  0.08563475 0.09556818 0.09817876
 0.06304395 0.07503274 0.04714863 0.01620013 0.01763884 0.07943611
 0.04862657 0.0587012  0.04898753 0.08289116 0.08570955 0.11513955
 0.10151444 0.03339988 0.02703426 0.07